# SR3 ATM Correlation Pair Screener

## Purpose
Grid-search **all viable SOFR cross-tenor pairs** to find the most consistently
mispriced around FOMC meetings. Uses the same signal logic as the backtest
(`market_cost − model_value`) but scans the full universe instead of hardcoded pairs.

## Universe
- **SFR** CM1–5 (quarterly SOFR)
- **0Q** CM1–3 (1Y midcurves)
- **2Q** CM1–3 (2Y midcurves)
- **3Q** CM1–2 (3Y midcurves)

13 instruments → C(13,2) = 78 raw pairs, ~60 after filtering.

## Liquidity Tiers
| Tier | Instruments | Description |
|------|------------|-------------|
| 1 | SFR CM1–3, 0Q CM1–2 | Most liquid quarterlies + near 1Y midcurves |
| 2 | SFR CM4–5, 0Q CM3, 2Q CM1–2 | Belly quarterlies + far 1Y MC + near 2Y MC |
| 3 | 2Q CM3, 3Q CM1–2 | Far 2Y MC + 3Y midcurves |

## Output
- `sr3_corr_screener_results.parquet` — one row per (pair × meeting)
- `sr3_corr_pair_rankings.parquet` — aggregated pair stats with consistency scores

In [ ]:
import sys
sys.path.append("..")

import datetime as dt
import math
import time
import os
from itertools import combinations
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import QuantLib as ql
from tqdm.auto import tqdm

from BT.misc import ql_cal_date_range
from MDP.STIRFutures.STIRFutureMDP import STIRFutureMDP
from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP
from Query.IRSwaps._CENTRAL_BANK_DATES import _CENTRAL_BANK_DATES

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
START_DATE = dt.date(2025, 1, 1)
END_DATE = dt.date(2026, 2, 27)

# Pair universe: (root, max_cm_rank)
UNIVERSE_SPEC = [
    ("SFR", 5),   # quarterly SOFR CM1-5
    ("0Q",  3),   # 1Y midcurve CM1-3
    ("2Q",  3),   # 2Y midcurve CM1-3
    ("3Q",  2),   # 3Y midcurve CM1-2
]

# Liquidity tier assignments: (root, cm_rank) -> tier
TIER_MAP = {}
# Tier 1: SFR CM1-3, 0Q CM1-2
for r in range(1, 4): TIER_MAP[("SFR", r)] = 1
for r in range(1, 3): TIER_MAP[("0Q", r)] = 1
# Tier 2: SFR CM4-5, 0Q CM3, 2Q CM1-2
for r in range(4, 6): TIER_MAP[("SFR", r)] = 2
TIER_MAP[("0Q", 3)] = 2
for r in range(1, 3): TIER_MAP[("2Q", r)] = 2
# Tier 3: 2Q CM3, 3Q CM1-2
TIER_MAP[("2Q", 3)] = 3
for r in range(1, 3): TIER_MAP[("3Q", r)] = 3

# Which tiers to scan (set to [1] for fast test, [1,2,3] for full scan)
SCAN_TIERS = [1, 2, 3]

# Timing around FOMC events
ENTRY_BDAYS_BEFORE_EVENT = 5

# Realized correlation parameters
REALIZED_LOOKBACK = 20
LONGRUN_LOOKBACK = 120

# SEP meeting months
SEP_MONTHS = {3, 6, 9, 12}

# Execution safety
MAX_EVENTS = None            # None = all FOMC in range
REQUEST_SLEEP_SECONDS = 0.25
REQUEST_RETRIES = 3
BATCH_SIZE = 10              # pairs between sleep pauses
BATCH_SLEEP = 2.0            # seconds between batches

# Output directory (relative to notebook)
OUTPUT_DIR = "."

In [ ]:
# ---------------------------------------------------------------------------
# Build pair universe
# ---------------------------------------------------------------------------
instruments = []
for root, max_rank in UNIVERSE_SPEC:
    for rank in range(1, max_rank + 1):
        alias = f"{root}CM{rank}"
        tier = TIER_MAP.get((root, rank), 99)
        instruments.append({"root": root, "rank": rank, "alias": alias, "tier": tier})

instr_df = pd.DataFrame(instruments)
print(f"Total instruments: {len(instr_df)}")
display(instr_df)

# Filter to selected tiers
active_instr = instr_df[instr_df["tier"].isin(SCAN_TIERS)].reset_index(drop=True)
print(f"\nActive instruments (tiers {SCAN_TIERS}): {len(active_instr)}")

# Generate all pairs from active instruments
raw_pairs = []
for (i, row_i), (j, row_j) in combinations(active_instr.iterrows(), 2):
    # Skip same-root same-rank (trivial)
    if row_i["alias"] == row_j["alias"]:
        continue
    # Assign pair tier = max tier of the two legs
    pair_tier = max(row_i["tier"], row_j["tier"])
    label = f"{row_i['alias']}_vs_{row_j['alias']}"
    # Pair type for grouping
    roots = tuple(sorted([row_i["root"], row_j["root"]]))
    if roots[0] == roots[1]:
        ptype = f"{roots[0]}x{roots[0]}"
    else:
        ptype = f"{roots[0]}x{roots[1]}"
    raw_pairs.append({
        "leg_i": row_i["alias"],
        "leg_j": row_j["alias"],
        "label": label,
        "root_i": row_i["root"],
        "root_j": row_j["root"],
        "rank_i": row_i["rank"],
        "rank_j": row_j["rank"],
        "tier": pair_tier,
        "pair_type": ptype,
    })

pairs_df = pd.DataFrame(raw_pairs)
print(f"\nRaw pairs: {len(pairs_df)}")
print(f"By tier: {pairs_df['tier'].value_counts().sort_index().to_dict()}")
print(f"By pair type: {pairs_df['pair_type'].value_counts().to_dict()}")

In [ ]:
# ---------------------------------------------------------------------------
# MDPs and calendar (same setup as backtest)
# ---------------------------------------------------------------------------
stirf_mdp = STIRFutureMDP(source="BARCHART_TOS_LIVE_STIRF-RL")
stirfo_mdp = STIRFutureOptionMDP(source="BARCHART_STIRFO-QL")

CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
_cal_start = START_DATE - dt.timedelta(days=int(LONGRUN_LOOKBACK * 2.0))
ALL_BDAYS = [
    pd.Timestamp(x).date()
    for x in ql_cal_date_range(
        ql_cal=CAL,
        start=dt.datetime.combine(_cal_start, dt.time()),
        end=dt.datetime.combine(END_DATE, dt.time()),
    )
]
BDAY_TO_INDEX = {d: i for i, d in enumerate(ALL_BDAYS)}


def nearest_prev_bday(d: dt.date) -> Optional[dt.date]:
    x = d
    while x not in BDAY_TO_INDEX:
        x -= dt.timedelta(days=1)
        if x < ALL_BDAYS[0]:
            return None
    return x


def shift_bday(d: dt.date, offset: int) -> Optional[dt.date]:
    d0 = nearest_prev_bday(d)
    if d0 is None:
        return None
    i = BDAY_TO_INDEX[d0] + int(offset)
    if 0 <= i < len(ALL_BDAYS):
        return ALL_BDAYS[i]
    return None


print(f"Calendar: {ALL_BDAYS[0]} to {ALL_BDAYS[-1]}  ({len(ALL_BDAYS)} bdays)")

In [ ]:
# ---------------------------------------------------------------------------
# Data fetching utilities (shared with backtest)
# ---------------------------------------------------------------------------
_option_cache: Dict[Tuple[dt.date, str], object] = {}
_future_cache: Dict[Tuple[dt.date, str], object] = {}


def _safe_first(v):
    if isinstance(v, list):
        return v[0] if v else None
    return v


def _retry_fetch(fetch_fn, retries=REQUEST_RETRIES):
    for i in range(retries):
        try:
            return fetch_fn()
        except Exception:
            if i == retries - 1:
                return {}
            time.sleep((i + 1) * 0.5)
    return {}


def get_option_pricers(as_of: dt.date, symbols: List[str]) -> Dict[str, object]:
    symbols = [str(s).upper() for s in symbols]
    missing = [s for s in symbols if (as_of, s) not in _option_cache]
    if missing:
        time.sleep(REQUEST_SLEEP_SECONDS)
        out = _retry_fetch(
            lambda: stirfo_mdp.get_data(
                {"endpoint": "option_snapshot", "symbols": missing,
                 "timestamp": as_of, "show_tqdm": False}
            )
        )
        for s in missing:
            _option_cache[(as_of, s)] = _safe_first(out.get(s))
    return {s: _option_cache.get((as_of, s)) for s in symbols}


def get_future_pricers(as_of: dt.date, symbols: List[str]) -> Dict[str, object]:
    symbols = [str(s).upper() for s in symbols]
    missing = [s for s in symbols if (as_of, s) not in _future_cache]
    if missing:
        time.sleep(REQUEST_SLEEP_SECONDS)
        out = _retry_fetch(
            lambda: stirf_mdp.get_data(
                {"symbols": missing, "timestamp": as_of, "show_tqdm": False}
            )
        )
        for s in missing:
            _future_cache[(as_of, s)] = _safe_first(out.get(s))
    return {s: _future_cache.get((as_of, s)) for s in symbols}


def _pricer_field(pr, field: str) -> float:
    if pr is None:
        return np.nan
    try:
        v = getattr(pr, field)()
        return float(v) if np.isfinite(v) else np.nan
    except Exception:
        return np.nan


def fetch_atm_leg(as_of: dt.date, cm_alias: str) -> Optional[dict]:
    """Fetch ATM call for a CM alias; return vol / price / underlying info."""
    sym = f"{cm_alias}|ATMC"
    pmap = get_option_pricers(as_of, [sym])
    pr = pmap.get(sym.upper())
    if pr is None:
        return None
    iv = _pricer_field(pr, "iv_normal")
    px = _pricer_field(pr, "price")
    if not (np.isfinite(iv) and iv > 0 and np.isfinite(px)):
        return None
    try:
        return {
            "symbol": pr.symbol(),
            "underlying_symbol": pr.underlying_symbol(),
            "call_price": px,
            "straddle_price": 2.0 * px,
            "iv_normal": iv,
            "iv_normal_bps": _pricer_field(pr, "iv_normal_bps"),
            "expiry_date": pr.expiry_date(),
        }
    except Exception:
        return None


def realized_corr(sym_i: str, sym_j: str, end_date: dt.date, lookback: int) -> float:
    """Trailing pairwise correlation of daily futures price changes."""
    end_bday = nearest_prev_bday(end_date)
    if end_bday is None:
        return np.nan
    i_end = BDAY_TO_INDEX[end_bday]
    i_start = i_end - lookback
    if i_start < 1:
        return np.nan
    dates = ALL_BDAYS[i_start - 1 : i_end + 1]
    rows = []
    for d in dates:
        pmap = get_future_pricers(d, [sym_i, sym_j])
        pi, pj = pmap.get(sym_i), pmap.get(sym_j)
        if pi is None or pj is None:
            continue
        rows.append((d, float(pi.price()), float(pj.price())))
    if len(rows) < max(10, lookback // 2):
        return np.nan
    df = pd.DataFrame(rows, columns=["date", "i", "j"]).set_index("date").sort_index()
    c = df["i"].diff().corr(df["j"].diff())
    return float(c) if np.isfinite(c) else np.nan


def spread_vol(sigma_i: float, sigma_j: float, rho: float) -> float:
    v = sigma_i**2 + sigma_j**2 - 2.0 * rho * sigma_i * sigma_j
    return math.sqrt(max(v, 0.0))


def bachelier_atm_straddle(sigma: float, T: float) -> float:
    return sigma * math.sqrt(max(T, 1e-6)) * math.sqrt(2.0 / math.pi)

In [ ]:
# ---------------------------------------------------------------------------
# FOMC events
# ---------------------------------------------------------------------------
fomc_events = sorted(
    {v[0] for v in _CENTRAL_BANK_DATES["USD-FEDFUNDS"].values()
     if START_DATE <= v[0] <= END_DATE}
)
if MAX_EVENTS is not None:
    fomc_events = fomc_events[:MAX_EVENTS]

print(f"FOMC meetings in range: {len(fomc_events)}")
for ev in fomc_events:
    tag = "SEP" if ev.month in SEP_MONTHS else ""
    print(f"  {ev}  {tag}")

In [ ]:
# ---------------------------------------------------------------------------
# Main screener loop: compute signal for every (pair x meeting)
# ---------------------------------------------------------------------------
rows = []
n_pairs = len(pairs_df)
n_events = len(fomc_events)
print(f"Scanning {n_pairs} pairs x {n_events} meetings = {n_pairs * n_events} combos")

for pair_idx, pair_row in tqdm(pairs_df.iterrows(), total=n_pairs, desc="Pairs"):
    leg_i_alias = pair_row["leg_i"]
    leg_j_alias = pair_row["leg_j"]
    pair_label = pair_row["label"]
    pair_tier = pair_row["tier"]
    pair_type = pair_row["pair_type"]

    for ev in fomc_events:
        entry_date = shift_bday(ev, -ENTRY_BDAYS_BEFORE_EVENT)
        if entry_date is None:
            continue

        # Fetch ATM option data on entry date
        leg_i = fetch_atm_leg(entry_date, leg_i_alias)
        leg_j = fetch_atm_leg(entry_date, leg_j_alias)
        if leg_i is None or leg_j is None:
            continue

        sigma_i = leg_i["iv_normal"]
        sigma_j = leg_j["iv_normal"]
        und_i = str(leg_i["underlying_symbol"])
        und_j = str(leg_j["underlying_symbol"])

        # Skip if both legs resolve to the same underlying
        if und_i == und_j:
            continue

        # Trailing realized rho
        rho_trail = realized_corr(und_i, und_j, entry_date, REALIZED_LOOKBACK)
        rho_longrun = realized_corr(und_i, und_j, entry_date, LONGRUN_LOOKBACK)
        if not np.isfinite(rho_trail):
            continue

        # Model rho
        is_sep = ev.month in SEP_MONTHS
        rho_base = rho_longrun if np.isfinite(rho_longrun) else rho_trail
        sep_adj = 0.03 if is_sep else -0.02
        rho_model = float(np.clip(rho_base + sep_adj, -0.999, 0.999))

        # Spread vol
        sigma_sp_model = spread_vol(sigma_i, sigma_j, rho_model)

        # Time to expiry
        exp_i = leg_i.get("expiry_date")
        exp_j = leg_j.get("expiry_date")
        T_i = max((exp_i - entry_date).days, 1) / 365.0 if exp_i else 0.1
        T_j = max((exp_j - entry_date).days, 1) / 365.0 if exp_j else 0.1
        T_avg = (T_i + T_j) / 2.0

        # Market cost vs model value
        ratio_i = sigma_j / (sigma_i + sigma_j)
        ratio_j = sigma_i / (sigma_i + sigma_j)
        market_cost = ratio_i * leg_i["straddle_price"] + ratio_j * leg_j["straddle_price"]
        model_value = bachelier_atm_straddle(sigma_sp_model, T_avg)
        signal = market_cost - model_value

        rows.append({
            "pair": pair_label,
            "pair_type": pair_type,
            "tier": pair_tier,
            "leg_i": leg_i_alias,
            "leg_j": leg_j_alias,
            "event_date": ev,
            "is_sep": is_sep,
            "entry_date": entry_date,
            "underlying_i": und_i,
            "underlying_j": und_j,
            "sigma_i": sigma_i,
            "sigma_j": sigma_j,
            "vol_i_bps": leg_i["iv_normal_bps"],
            "vol_j_bps": leg_j["iv_normal_bps"],
            "straddle_px_i": leg_i["straddle_price"],
            "straddle_px_j": leg_j["straddle_price"],
            "rho_trail": rho_trail,
            "rho_longrun": rho_longrun if np.isfinite(rho_longrun) else np.nan,
            "rho_model": rho_model,
            "sigma_sp_model": sigma_sp_model,
            "T_avg": T_avg,
            "market_cost": market_cost,
            "model_value": model_value,
            "signal": signal,
            "signal_abs": abs(signal),
        })

    # Batch sleep to avoid 429s
    if (pair_idx + 1) % BATCH_SIZE == 0:
        time.sleep(BATCH_SLEEP)

screener_df = pd.DataFrame(rows)
print(f"\nTotal screener results: {len(screener_df)}")
if not screener_df.empty:
    print(f"Unique pairs with data: {screener_df['pair'].nunique()}")
    print(f"Unique events with data: {screener_df['event_date'].nunique()}")
    print(f"By tier: {screener_df.groupby('tier')['pair'].nunique().to_dict()}")

In [ ]:
# ---------------------------------------------------------------------------
# Per-meeting ranking: rank pairs by |signal| within each meeting
# ---------------------------------------------------------------------------
if not screener_df.empty:
    screener_df["signal_rank"] = screener_df.groupby("event_date")["signal_abs"].rank(
        ascending=False, method="min"
    ).astype(int)

    print("Top 5 pairs per meeting (by |signal|):")
    for ev in sorted(screener_df["event_date"].unique()):
        sub = screener_df[screener_df["event_date"] == ev].nsmallest(5, "signal_rank")
        tag = "SEP" if sub["is_sep"].iloc[0] else ""
        print(f"\n  {ev} {tag}:")
        for _, r in sub.iterrows():
            print(f"    #{r['signal_rank']:2d}  {r['pair']:30s}  signal={r['signal']:+.6f}  "
                  f"rho_trail={r['rho_trail']:.3f}  tier={r['tier']}")
else:
    print("No screener results.")

In [ ]:
# ---------------------------------------------------------------------------
# Cross-meeting consistency scoring
# ---------------------------------------------------------------------------
if not screener_df.empty:
    pair_stats = (
        screener_df.groupby(["pair", "pair_type", "tier", "leg_i", "leg_j"])
        .agg(
            n_events=("signal", "count"),
            mean_signal=("signal", "mean"),
            std_signal=("signal", "std"),
            mean_abs_signal=("signal_abs", "mean"),
            median_abs_signal=("signal_abs", "median"),
            max_abs_signal=("signal_abs", "max"),
            mean_rho_trail=("rho_trail", "mean"),
            std_rho_trail=("rho_trail", "std"),
            pct_positive=("signal", lambda x: (x > 0).mean()),
        )
        .reset_index()
    )

    # Consistency score: mean_|signal| / std_signal * sqrt(n_events)
    # Higher = more consistently mispriced across meetings
    pair_stats["consistency_score"] = (
        pair_stats["mean_abs_signal"]
        / pair_stats["std_signal"].replace(0, np.nan)
        * np.sqrt(pair_stats["n_events"])
    )

    # Signal direction stability: how often does the signal point the same way?
    pair_stats["direction_stability"] = (
        2 * (pair_stats["pct_positive"] - 0.5).abs()
    )  # 0 = random, 1 = always same direction

    pair_stats = pair_stats.sort_values("consistency_score", ascending=False).reset_index(drop=True)

    print("=== Pair Rankings (by consistency score) ===")
    display(pair_stats.head(20)[
        ["pair", "pair_type", "tier", "n_events", "mean_abs_signal",
         "std_signal", "consistency_score", "direction_stability",
         "mean_rho_trail", "pct_positive"]
    ])
else:
    pair_stats = pd.DataFrame()

In [ ]:
# ---------------------------------------------------------------------------
# Visualization 1: Signal heatmap (pair x meeting)
# ---------------------------------------------------------------------------
if not screener_df.empty:
    # Use top 30 pairs by consistency for readability
    top_pairs = pair_stats.head(30)["pair"].tolist()
    hm_data = screener_df[screener_df["pair"].isin(top_pairs)].copy()

    pivot = hm_data.pivot_table(
        index="pair", columns="event_date", values="signal", aggfunc="first"
    )
    # Reorder rows by consistency ranking
    pivot = pivot.reindex(top_pairs)

    fig, ax = plt.subplots(figsize=(14, max(8, len(top_pairs) * 0.35)))
    vmax = max(abs(pivot.min().min()), abs(pivot.max().max())) if not pivot.empty else 1
    sns.heatmap(
        pivot, center=0, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
        linewidths=0.5, annot=True, fmt=".4f", annot_kws={"fontsize": 7},
        ax=ax
    )
    ax.set_title("Signal Heatmap (top 30 pairs by consistency, market_cost - model_value)")
    ax.set_ylabel("Pair")
    ax.set_xlabel("FOMC Meeting Date")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Visualization 2: Consistency bar chart + supplementary plots
# ---------------------------------------------------------------------------
if not pair_stats.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    # 1. Consistency score bar chart (top 20)
    ax = axes[0, 0]
    top20 = pair_stats.head(20)
    colors = top20["tier"].map({1: "#2196F3", 2: "#FF9800", 3: "#9C27B0"}).fillna("gray")
    ax.barh(range(len(top20)), top20["consistency_score"].values, color=colors)
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20["pair"].values, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Consistency Score")
    ax.set_title("Top 20 Pairs by Consistency Score")
    # Legend for tiers
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#2196F3", label="Tier 1"),
        Patch(facecolor="#FF9800", label="Tier 2"),
        Patch(facecolor="#9C27B0", label="Tier 3"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=7)
    ax.grid(alpha=0.3, axis="x")

    # 2. Signal persistence scatter (mean_abs_signal vs direction_stability)
    ax = axes[0, 1]
    for t in sorted(pair_stats["tier"].unique()):
        sub = pair_stats[pair_stats["tier"] == t]
        ax.scatter(
            sub["mean_abs_signal"], sub["direction_stability"],
            label=f"Tier {t}", alpha=0.6, s=40
        )
    ax.set_xlabel("|Signal| Mean")
    ax.set_ylabel("Direction Stability (0=random, 1=persistent)")
    ax.set_title("Signal Magnitude vs Persistence")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 3. Pair type distribution of consistency scores
    ax = axes[1, 0]
    ptypes = pair_stats["pair_type"].unique()
    bp_data = [pair_stats[pair_stats["pair_type"] == pt]["consistency_score"].dropna().values
               for pt in ptypes]
    bp = ax.boxplot(bp_data, labels=ptypes, patch_artist=True)
    cmap_bp = plt.cm.Set2(np.linspace(0, 1, len(ptypes)))
    for patch, c in zip(bp["boxes"], cmap_bp):
        patch.set_facecolor(c)
    ax.set_ylabel("Consistency Score")
    ax.set_title("Consistency by Pair Type")
    ax.grid(alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    # 4. Tier vs |signal| boxplots
    ax = axes[1, 1]
    tier_data = [screener_df[screener_df["tier"] == t]["signal_abs"].values
                 for t in sorted(screener_df["tier"].unique())]
    tier_labels = [f"Tier {t}" for t in sorted(screener_df["tier"].unique())]
    bp2 = ax.boxplot(tier_data, labels=tier_labels, patch_artist=True)
    tier_colors = ["#2196F3", "#FF9800", "#9C27B0"]
    for patch, c in zip(bp2["boxes"], tier_colors[:len(bp2["boxes"])]):
        patch.set_facecolor(c)
    ax.set_ylabel("|Signal|")
    ax.set_title("|Signal| Distribution by Liquidity Tier")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Visualization 3: Consistently mispriced vs one-off outliers
# ---------------------------------------------------------------------------
if not pair_stats.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Consistently mispriced: high consistency, high direction stability
    ax = axes[0]
    cs = pair_stats.dropna(subset=["consistency_score"]).copy()
    cs["category"] = "Moderate"
    cs.loc[
        (cs["consistency_score"] > cs["consistency_score"].quantile(0.75)) &
        (cs["direction_stability"] > 0.5),
        "category"
    ] = "Consistent"
    cs.loc[
        (cs["max_abs_signal"] > cs["max_abs_signal"].quantile(0.9)) &
        (cs["direction_stability"] < 0.3),
        "category"
    ] = "One-off Outlier"

    colors_cat = {"Consistent": "#4CAF50", "One-off Outlier": "#F44336", "Moderate": "#9E9E9E"}
    for cat in ["Moderate", "One-off Outlier", "Consistent"]:
        sub = cs[cs["category"] == cat]
        ax.scatter(
            sub["mean_abs_signal"], sub["consistency_score"],
            c=colors_cat[cat], label=f"{cat} ({len(sub)})", alpha=0.7, s=50
        )
    # Label top consistent pairs
    top_consistent = cs[cs["category"] == "Consistent"].nlargest(5, "consistency_score")
    for _, r in top_consistent.iterrows():
        ax.annotate(r["pair"], (r["mean_abs_signal"], r["consistency_score"]),
                    fontsize=6, alpha=0.8)
    ax.set_xlabel("|Signal| Mean")
    ax.set_ylabel("Consistency Score")
    ax.set_title("Consistently Mispriced vs One-off Outliers")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 2. Average rho by pair type
    ax = axes[1]
    rho_by_type = pair_stats.groupby("pair_type").agg(
        mean_rho=("mean_rho_trail", "mean"),
        std_rho=("mean_rho_trail", "std"),
        n=("pair", "count"),
    ).sort_values("mean_rho", ascending=False)
    ax.bar(range(len(rho_by_type)), rho_by_type["mean_rho"], yerr=rho_by_type["std_rho"],
           capsize=4, color="steelblue", alpha=0.7)
    ax.set_xticks(range(len(rho_by_type)))
    ax.set_xticklabels([f"{idx}\n(n={row['n']})" for idx, row in rho_by_type.iterrows()],
                       fontsize=8)
    ax.set_ylabel("Mean Trailing Correlation")
    ax.set_title("Average Correlation by Pair Type")
    ax.axhline(0, color="k", lw=0.5)
    ax.grid(alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Export results to parquet
# ---------------------------------------------------------------------------
if not screener_df.empty:
    results_path = os.path.join(OUTPUT_DIR, "sr3_corr_screener_results.parquet")
    screener_df.to_parquet(results_path, index=False)
    print(f"Saved screener results: {results_path} ({len(screener_df)} rows)")

if not pair_stats.empty:
    rankings_path = os.path.join(OUTPUT_DIR, "sr3_corr_pair_rankings.parquet")
    pair_stats.to_parquet(rankings_path, index=False)
    print(f"Saved pair rankings: {rankings_path} ({len(pair_stats)} rows)")

    print(f"\n=== Top 10 Most Consistently Mispriced Pairs ===")
    for i, r in pair_stats.head(10).iterrows():
        print(f"  {i+1:2d}. {r['pair']:30s}  tier={r['tier']}  "
              f"consistency={r['consistency_score']:.3f}  "
              f"mean_|sig|={r['mean_abs_signal']:.6f}  "
              f"dir_stab={r['direction_stability']:.2f}  "
              f"n={r['n_events']}")